# Customer 360 Exploratory Analysis
**Snowflake Notebook** — Analytical companion to the Streamlit operational dashboards.

Analyses: segmentation, churn risk factors, health scores, revenue at risk, NBA effectiveness, document intelligence, AI quality.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
session = get_active_session()
print('Connected to Snowflake')

## 1. Customer Segment Profiling

In [ ]:
segments = session.sql("""
SELECT CUSTOMER_SEGMENT, COUNT(*) AS CUSTOMERS,
  ROUND(AVG(CREDIT_SCORE),0) AS AVG_CREDIT, ROUND(AVG(ANNUAL_INCOME),0) AS AVG_INCOME,
  ROUND(AVG(TOTAL_PREMIUM),0) AS AVG_PREMIUM, ROUND(AVG(ESTIMATED_CLV),0) AS AVG_CLV,
  ROUND(100.0*SUM(CASE WHEN IS_ACTIVE THEN 1 ELSE 0 END)/COUNT(*),1) AS ACTIVE_PCT
FROM CUSTOMER_360.CURATED.CUSTOMER_360_UNIFIED GROUP BY 1 ORDER BY AVG_CLV DESC
""").to_pandas()
segments

## 2. Churn Risk Factor Analysis

In [ ]:
risk = session.sql("""
SELECT CASE WHEN cr.CHURN_RISK_SCORE>=0.6 THEN 'High Risk' ELSE 'Low Risk' END AS RISK_GROUP,
  ROUND(AVG(c.COMPLAINT_COUNT),2) AS AVG_COMPLAINTS,
  ROUND(AVG(c.MAX_DAYS_PAST_DUE),0) AS AVG_DPD,
  ROUND(AVG(c.LAPSED_POLICIES),2) AS AVG_LAPSED,
  ROUND(AVG(c.DAYS_SINCE_LAST_INTERACTION),0) AS AVG_DAYS_NO_CONTACT,
  COUNT(*) AS CUSTOMERS
FROM CUSTOMER_360.CURATED.CUSTOMER_360_UNIFIED c
JOIN CUSTOMER_360.AI.DT_CHURN_RISK cr ON c.CUSTOMER_ID=cr.CUSTOMER_ID
WHERE cr.CHURN_RISK_SCORE IS NOT NULL GROUP BY 1
""").to_pandas()
risk

## 3. Health Score Distribution

In [ ]:
health = session.sql("""
SELECT HEALTH_GRADE, CUSTOMER_SEGMENT, COUNT(*) AS CUSTOMERS, ROUND(AVG(HEALTH_SCORE),1) AS AVG_SCORE
FROM CUSTOMER_360.AI.DT_CUSTOMER_HEALTH_SCORE
GROUP BY 1,2 ORDER BY 1,2
""").to_pandas()
health

## 4. Revenue at Risk

In [ ]:
revenue = session.sql("""
SELECT cr.RETENTION_URGENCY, COUNT(*) AS CUSTOMERS,
  ROUND(SUM(c.TOTAL_PREMIUM),0) AS PREMIUM_AT_RISK,
  ROUND(SUM(c.ESTIMATED_CLV),0) AS CLV_AT_RISK
FROM CUSTOMER_360.CURATED.CUSTOMER_360_UNIFIED c
JOIN CUSTOMER_360.AI.DT_CHURN_RISK cr ON c.CUSTOMER_ID=cr.CUSTOMER_ID
WHERE cr.CHURN_RISK_SCORE>=0.4 GROUP BY 1 ORDER BY CLV_AT_RISK DESC
""").to_pandas()
revenue

## 5. AI Quality Metrics

In [ ]:
quality = session.sql('SELECT * FROM CUSTOMER_360.APP.AI_QUALITY_DASHBOARD').to_pandas()
quality